In [0]:
import os, re, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def one(sql):
    return spark.sql(sql).first()[0]

In [0]:
tmp = privacy.create_functions(spark, cfg, env, prefix="tmp_mask_", clear=["lh_nobody"])
f = {n.split(".")[-1]: n for n in tmp}

check("last4", one(f"SELECT {f['tmp_mask_last4']}('123456789012')") == "XXXXXXXX9012")
check("last4 short value", one(f"SELECT {f['tmp_mask_last4']}('ab')") == "XX")
check("last4 keeps NULL", one(f"SELECT {f['tmp_mask_last4']}(CAST(NULL AS STRING))") is None)
check("initial", one(f"SELECT {f['tmp_mask_initial']}('Test Person')") == "T***")
t1 = one(f"SELECT {f['tmp_mask_token']}('PAY0001971')")
t2 = one(f"SELECT {f['tmp_mask_token']}('PAY0001972')")
check("token shape", bool(re.match(r"^PAY#[0-9A-F]{8}$", t1)), t1)
check("token same input, same token", t1 == one(f"SELECT {f['tmp_mask_token']}('PAY0001971')"))
check("token other input, other token", t1 != t2)
check("hide text", one(f"SELECT {f['tmp_mask_hide_string']}('9876543210')") is None)
check("hide date", one(f"SELECT {f['tmp_mask_hide_date']}(DATE'1990-05-17')") is None)

mine = [g for g in ["lh_developer", "lh_teamlead", "lh_manager", "lh_bizmanager", "lh_management"]
        if one(f"SELECT is_account_group_member('{g}')")]
print("INFO your groups:", mine)
clear_here = bool(set(mine) & set(privacy.load()["clear"][env]))
seen = one(f"SELECT {privacy.function_name(cfg, 'last4')}('123456789012')")
check("real function follows your groups", seen == ("123456789012" if clear_here else "XXXXXXXX9012"), seen)

In [0]:
probe = f"{cat}.ops.tmp_mask_probe"
source = f"{cat}.ops.tmp_mask_probe_src"
ckpt = f"/Volumes/{cat}/ops/checkpoints/tmp_mask_probe"

def attempt(name, fn, must=True):
    try:
        ok, detail = True, str(fn())
    except Exception as e:
        ok, detail = False, str(e).split("\n")[0][:160]
    if must:
        check(name, ok, detail)
    else:
        print("INFO", name, "->", "WORKS" if ok else "NOT SUPPORTED", detail)

def same(got, wanted):
    if got != wanted:
        raise ValueError(f"got {got}")
    return got

def stream_append():
    spark.sql(f"CREATE OR REPLACE TABLE {source} AS SELECT 'E' AS id, '555566667777' AS secret")
    (spark.readStream.table(source).writeStream.option("checkpointLocation", ckpt)
          .trigger(availableNow=True).toTable(probe).awaitTermination())
    return same(spark.table(probe).filter("id = 'E'").select("secret").first()[0], "XXXXXXXX7777")

def restore():
    spark.sql(f"RESTORE TABLE {probe} TO VERSION AS OF 3")
    return same(spark.table(probe).count(), 3)

def overwrite():
    (spark.createDataFrame([("Z", "121212121212", 1)], "id string, secret string, extra int")
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(probe))
    masks = one(f"""SELECT count(*) FROM {cat}.information_schema.column_masks
                    WHERE table_schema = 'ops' AND table_name = 'tmp_mask_probe'""")
    return "mask kept" if masks else "MASK LOST"

spark.sql(f"CREATE OR REPLACE TABLE {probe} (id STRING, secret STRING)")                    # version 0
spark.sql(f"INSERT INTO {probe} VALUES ('A', '111122223333'), ('B', '444455556666')")      # version 1
attempt("set mask on a column",
        lambda: spark.sql(f"ALTER TABLE {probe} ALTER COLUMN secret SET MASK {f['tmp_mask_last4']}").count())   # version 2
attempt("read is masked",
        lambda: same([r[0] for r in spark.sql(f"SELECT secret FROM {probe} ORDER BY id").collect()],
                     ["XXXXXXXX3333", "XXXXXXXX6666"]))
attempt("MERGE into a masked table", lambda: spark.sql(f"""
        MERGE INTO {probe} t
        USING (SELECT 'A' AS id, '999988887777' AS secret UNION ALL SELECT 'C', '000011112222') s
        ON t.id = s.id
        WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""").count())           # version 3
attempt("append to a masked table",
        lambda: spark.createDataFrame([("D", "123412341234")], "id string, secret string")
                     .write.mode("append").saveAsTable(probe))                              # version 4
attempt("new values stored, read still masked",
        lambda: same(sorted(r[0] for r in spark.table(probe).select("secret").collect()),
                     ["XXXXXXXX1234", "XXXXXXXX2222", "XXXXXXXX6666", "XXXXXXXX7777"]))
attempt("stream append (as Auto Loader does)", stream_append, must=False)
attempt("time travel (VERSION AS OF)",
        lambda: spark.sql(f"SELECT count(*) FROM {probe} VERSION AS OF 1").first()[0], must=False)
attempt("RESTORE to an older version", restore, must=False)
attempt("full overwrite with a new schema", overwrite, must=False)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {probe}")
spark.sql(f"DROP TABLE IF EXISTS {source}")
dbutils.fs.rm(ckpt, True)
for n in tmp:
    spark.sql(f"DROP FUNCTION IF EXISTS {n}")
left = [r[0] for r in spark.sql(
            f"SELECT routine_name FROM {cat}.information_schema.routines WHERE routine_schema = 'ops'").collect()
        if r[0].startswith("tmp_mask_")]
check("temporary functions removed", not left, str(left))
print(f"\nt14_mask_functions: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t14_mask_functions failed")